In [ ]:
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
!which ollama || echo 'NOT ON PATH'

In [ ]:
import subprocess, time, urllib.request

subprocess.Popen(['ollama', 'serve'])
for _ in range(30):
    try:
        urllib.request.urlopen('http://localhost:11434/api/version', timeout=2)
        print('Ollama server is up')
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('Ollama server did not come up in time')

In [ ]:
!git clone https://github.com/orceal-lab/cot-failure-modes.git
%cd cot-failure-modes
!pip install -q -r requirements.txt

In [ ]:
!ollama pull mistral:7b-instruct-q4_K_M
!ollama pull qwen2.5-coder:7b-instruct-q4_K_M

In [ ]:
!python run_experiment.py --problems problems/arithmetic_steps.json --model \
  "ollama:mistral:7b-instruct-q4_K_M" \
  "ollama:qwen2.5-coder:7b-instruct-q4_K_M"

!python run_experiment.py --problems problems/arithmetic_independent.json --model \
  "ollama:mistral:7b-instruct-q4_K_M" \
  "ollama:qwen2.5-coder:7b-instruct-q4_K_M"

In [ ]:
print("=== arithmetic_steps (running total) ===")
!python analyze.py results/raw_*_arithmetic_steps_*.json --results-dir results/steps
!python compare_models.py results/steps/analysis_*.csv --results-dir results/steps

print("\n=== arithmetic_independent (no running total) ===")
!python analyze.py results/raw_*_arithmetic_independent_*.json --results-dir results/independent
!python compare_models.py results/independent/analysis_*.csv --results-dir results/independent

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/results', 'zip', 'results')
print('Zipped results to /kaggle/working/results.zip')